# Bhop: RL Discovery of Bunnyhopping

Analysis notebook for examining trained PPO agents that learn to bunnyhop
using a faithful reimplementation of Quake III Arena's movement physics.

In [ ]:
import sys
sys.path.insert(0, "../src")
sys.path.insert(0, "../scripts")

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO

import bhop  # noqa: F401 -- triggers env registration
from bhop.viz import (
    plot_speed_over_time,
    plot_trajectory,
    plot_action_distribution,
    analyze_policy,
    plot_policy_heatmap,
)
from evaluate import run_episode

%matplotlib inline

In [ ]:
# Load environment and trained model
# Using the 2M-step continuous action space model
MODEL_PATH = "../models/bhop_2m_continuous"

env = gym.make("bhop/BhopFlat-v0")
env.reset(seed=42)
model = PPO.load(MODEL_PATH)

print(f"Model loaded from {MODEL_PATH}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

## Training Curves

Load TensorBoard event data from all hyperparameter sweep runs and plot
mean speed over training timesteps. Each line is one sweep configuration.

In [ ]:
import os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

RUNS_DIR = "../runs"

# Collect mean_speed and max_speed curves from all sweep runs
sweep_data = {}
for run_name in sorted(os.listdir(RUNS_DIR)):
    run_path = os.path.join(RUNS_DIR, run_name)
    if not os.path.isdir(run_path) or not run_name.startswith("ent"):
        continue
    ea = EventAccumulator(run_path)
    ea.Reload()
    scalars = ea.Tags().get("scalars", [])
    if "bhop/mean_speed" not in scalars:
        continue
    mean_events = ea.Scalars("bhop/mean_speed")
    max_events = ea.Scalars("bhop/max_speed")
    # Strip trailing _1 from run name for cleaner labels
    label = run_name.removesuffix("_1")
    sweep_data[label] = {
        "steps": [e.step for e in mean_events],
        "mean_speed": [e.value for e in mean_events],
        "max_speed": [e.value for e in max_events],
    }

print(f"Loaded {len(sweep_data)} sweep runs")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, data in sweep_data.items():
    ax1.plot(data["steps"], data["mean_speed"], ".-", label=name, markersize=4)
    ax2.plot(data["steps"], data["max_speed"], ".-", label=name, markersize=4)

ax1.axhline(320, color="r", linestyle="--", linewidth=0.8, label="sv_maxspeed (320)")
ax1.set_xlabel("Timesteps")
ax1.set_ylabel("Mean Episode Speed (ups)")
ax1.set_title("Mean Speed Over Training")
ax1.legend(fontsize=7, ncol=2)

ax2.axhline(320, color="r", linestyle="--", linewidth=0.8, label="sv_maxspeed (320)")
ax2.set_xlabel("Timesteps")
ax2.set_ylabel("Max Episode Speed (ups)")
ax2.set_title("Max Speed Over Training")
ax2.legend(fontsize=7, ncol=2)

fig.suptitle("Hyperparameter Sweep: Training Curves (10k timesteps)", fontsize=13)
fig.tight_layout()

os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/training_curves.png", dpi=150)
plt.show()

# Annotate: best final mean speed
best_name = max(sweep_data, key=lambda k: sweep_data[k]["mean_speed"][-1])
best_mean = sweep_data[best_name]["mean_speed"][-1]
best_max = sweep_data[best_name]["max_speed"][-1]
print(f"Best config: {best_name}")
print(f"  Final mean speed: {best_mean:.1f} ups")
print(f"  Final max speed:  {best_max:.1f} ups")
print(f"\nNote: These are short 10k-step runs. Longer training (2M-10M steps)")
print(f"is needed to reach 400+ ups and discover bunnyhopping.")

## Random vs Trained Agent

Run one episode with random actions and one with the trained model, then
compare speed over time and top-down trajectory side by side.

In [ ]:
# Run one episode with random actions
random_env = gym.make("bhop/BhopFlat-v0")
obs, _ = random_env.reset(seed=42)
random_data = {"speeds": [], "positions": [], "actions": [], "on_ground": []}

done = False
while not done:
    action = random_env.action_space.sample()
    obs, _, terminated, truncated, info = random_env.step(action)
    done = terminated or truncated
    random_data["speeds"].append(info["speed"])
    pos = random_env.unwrapped._physics.position[:2].copy()
    random_data["positions"].append(pos.tolist())
    random_data["actions"].append(action.copy())
    random_data["on_ground"].append(bool(random_env.unwrapped._physics.on_ground))
random_env.close()

# Run one episode with the trained model (stochastic -- deterministic policy
# is fragile at action thresholds with continuous actions)
trained_env = gym.make("bhop/BhopFlat-v0")
trained_env.reset(seed=42)
trained_data = run_episode(model, trained_env, deterministic=False)
trained_env.close()

print(f"Random agent  -- mean: {np.mean(random_data['speeds']):.1f}, max: {np.max(random_data['speeds']):.1f} ups")
print(f"Trained agent -- mean: {np.mean(trained_data['speeds']):.1f}, max: {np.max(trained_data['speeds']):.1f} ups")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Top row: speed over time
for ax, data, label in [(axes[0, 0], random_data, "Random Agent"), (axes[0, 1], trained_data, "Trained Agent")]:
    ax.plot(data["speeds"], linewidth=0.6)
    ax.axhline(320, color="r", linestyle="--", linewidth=0.8, label="sv_maxspeed (320)")
    ax.set_xlabel("Tick")
    ax.set_ylabel("Speed (ups)")
    ax.set_title(f"{label}: Speed Over Time")
    ax.legend(fontsize=8)

# Use same y-axis scale for fair comparison
max_y = max(max(random_data["speeds"]), max(trained_data["speeds"])) * 1.1
axes[0, 0].set_ylim(0, max_y)
axes[0, 1].set_ylim(0, max_y)

# Bottom row: top-down trajectory
for ax, data, label in [(axes[1, 0], random_data, "Random Agent"), (axes[1, 1], trained_data, "Trained Agent")]:
    positions = np.array(data["positions"])
    ax.plot(positions[:, 0], positions[:, 1], linewidth=0.4)
    ax.plot(positions[0, 0], positions[0, 1], "go", markersize=6, label="Start")
    ax.plot(positions[-1, 0], positions[-1, 1], "rs", markersize=6, label="End")
    ax.set_xlabel("X (units)")
    ax.set_ylabel("Y (units)")
    ax.set_title(f"{label}: Trajectory")
    ax.set_aspect("equal")
    ax.legend(fontsize=8)

fig.suptitle("Random vs Trained Agent Comparison", fontsize=13)
fig.tight_layout()
fig.savefig("../figures/random_vs_trained.png", dpi=150)
plt.show()

## Policy Analysis

Query the trained model's deterministic action at each speed for both ground and air states.
Compare the learned yaw delta against the theoretical optimal `arctan(320/speed)`, and check
whether the agent learned to always jump on ground and strafe in the air.

In [ ]:
policy_data = analyze_policy(model)

print(f"Ground jump %: {policy_data['ground_jump_pct']:.1f}%")
print(f"Speed range queried: {policy_data['speeds'][0]:.0f} - {policy_data['speeds'][-1]:.0f} ups ({len(policy_data['speeds'])} points)")

plot_policy_heatmap(policy_data)

# Show the saved figure inline
from IPython.display import Image, display
display(Image(filename="../figures/policy_heatmap.png"))